In [ ]:
import random
import pandas as pd
from time import time
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import (
    StratifiedKFold, RandomizedSearchCV, GridSearchCV, ParameterGrid, cross_val_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
 
RT = 42
N_ITER = 120
DATA_DIR = "/Users/fangsiyu/Desktop/SDU-2026-fall-code/AML/kaggle"
 

In [ ]:
X_test = pd.read_csv(f"{DATA_DIR}/X_test.csv")
X_train = pd.read_csv(f"{DATA_DIR}/X_train.csv")
y_train = pd.read_csv(f"{DATA_DIR}/y_train.csv")
 
X_test["Gender"] = X_test["Gender"].map({"Female": 0, "Male": 1})
X_train["Gender"] = X_train["Gender"].map({"Female": 0, "Male": 1})
y_train["Diabetes"] = y_train["Diabetes"].map({"No": 0, "Yes": 1})
 
X_train = X_train.set_index("ID")
X_test = X_test.set_index("ID")
y_train = y_train.set_index("ID")["Diabetes"]
 

In [ ]:
RISK_COLS = ["HighBP", "HighChol", "Stroke", "HeartDiseaseorAttack", "DiffWalk"]


def add_risk(X, bmi_cut=30):
    X = X.copy()
    X["RiskCount"] = X[RISK_COLS].sum(axis=1) + (X["BMI"] >= bmi_cut).astype(int)
    return X


def add_healthy(X):
    X = X.copy()
    X["HealthyCount"] = X[["PhysActivity", "Fruits", "Veggies"]].sum(axis=1)
    return X


def add_risk_healthy(X, bmi_cut=30):
    return add_healthy(add_risk(X, bmi_cut))


def gb_with(func):
    return Pipeline([
        ("feat", FunctionTransformer(func)),
        ("clf", GradientBoostingClassifier(random_state=RT)),
    ])


GB_GRID = {
    "learning_rate": [0.03, 0.05, 0.06, 0.1],
    "n_estimators": [150, 175, 200, 250, 300],
    "max_depth": [3, 4, 5, 6],
    "min_samples_leaf": [20, 50, 100, 150, 200],
    "subsample": [0.7, 0.8, 0.9, 1.0],
}
PIPE_GRID = {f"clf__{k}": v for k, v in GB_GRID.items()}

models = {
    "GB raw": (GradientBoostingClassifier(random_state=RT), GB_GRID),
    "GB + RiskCount": (gb_with(add_risk), PIPE_GRID),
    "GB + HealthyCount": (gb_with(add_healthy), PIPE_GRID),
    "GB + RiskCount + HealthyCount": (gb_with(add_risk_healthy), PIPE_GRID),
}

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RT)

best_models, rows = {}, []
for name, (model, grid) in models.items():
    t0 = time()
    n_combos = len(ParameterGrid(grid))
    common = dict(cv=cv, scoring="accuracy", n_jobs=-1)
    if n_combos > N_ITER:
        search = RandomizedSearchCV(model, grid, n_iter=N_ITER, random_state=RT, **common)
    else:
        search = GridSearchCV(model, grid, **common)
    search.fit(X_train, y_train)

    best_models[name] = search.best_estimator_
    rows.append({
        "Model": name,
        "Tried": f"{min(n_combos, N_ITER)}/{n_combos}",
        "CV Acc": search.best_score_,
        "CV Acc std": search.cv_results_["std_test_score"][search.best_index_],
        "Time (s)": round(time() - t0),
        "Best params": {k.split("__")[-1]: v for k, v in search.best_params_.items()},
    })
    print(f"{name} done, {rows[-1]['Time (s)']} sec")

grid_results = pd.DataFrame(rows)
print(grid_results.drop(columns="Best params").round(4).to_string(index=False))
for _, r in grid_results.iterrows():
    print(f"{r['Model']}: {r['Best params']}")

In [ ]:
# GB + HealthyCount done, 545 sec
#                         Model    Tried  CV Acc  CV Acc std  Time (s)
#                        GB raw 120/1600  0.7587      0.0024       512
#                GB + RiskCount 120/1600  0.7594      0.0024       544
# GB + RiskCount + HealthyCount 120/1600  0.7588      0.0032       587
#             GB + HealthyCount 120/1600  0.7582      0.0018       545
# GB raw: {'subsample': 0.8, 'n_estimators': 200, 'min_samples_leaf': 200, 'max_depth': 5, 'learning_rate': 0.03}
# GB + RiskCount: {'subsample': 0.9, 'n_estimators': 250, 'min_samples_leaf': 150, 'max_depth': 6, 'learning_rate': 0.03}
# GB + RiskCount + HealthyCount: {'subsample': 0.9, 'n_estimators': 250, 'min_samples_leaf': 100, 'max_depth': 6, 'learning_rate': 0.03}
# GB + HealthyCount: {'subsample': 0.9, 'n_estimators': 250, 'min_samples_leaf': 150, 'max_depth': 6, 'learning_rate': 0.03}

In [ ]:
random.seed(RT)
SEEDS = random.sample(range(1, 1001), 20)

paired_rows = []
for s in SEEDS:
    cv_s = StratifiedKFold(n_splits=5, shuffle=True, random_state=s)
    row = {"seed": s}
    for name, model in best_models.items():
        row[name] = cross_val_score(clone(model), X_train, y_train,
                                    cv=cv_s, scoring="accuracy", n_jobs=-1).mean()
    paired_rows.append(row)

paired = pd.DataFrame(paired_rows).set_index("seed")

PAIRS = [
    ("GB raw", "GB + RiskCount"),
    ("GB raw", "GB + HealthyCount"),
    ("GB raw", "GB + RiskCount + HealthyCount"),
    ("GB + RiskCount", "GB + RiskCount + HealthyCount"),
    ("GB + HealthyCount", "GB + RiskCount + HealthyCount"),
]

summary = pd.DataFrame([
    {
        "comparison": f"{b}  vs  {a}",
        "mean diff": (paired[b] - paired[a]).mean(),
        "wins": f"{((paired[b] - paired[a]) > 0).sum()}/{len(paired)}",
    }
    for a, b in PAIRS
])

print(paired.round(4).to_string())
print()
print(summary.round(4).to_string(index=False))

# %%
FILE_TAGS = {
    "GB raw": "gb_raw",
    "GB + RiskCount": "gb_riskcount",
    "GB + HealthyCount": "gb_healthy",
    "GB + RiskCount + HealthyCount": "gb_riskcount_healthy",
}

for name, model in best_models.items():
    final_model = clone(model).fit(X_train, y_train)
    pred = final_model.predict(X_test)

    submission = pd.DataFrame({"ID": X_test.index, "Diabetes": pred})
    submission["Diabetes"] = submission["Diabetes"].map({0: "No", 1: "Yes"})
    submission.to_csv(f"submission_{FILE_TAGS[name]}.csv", index=False)

    print(name, submission["Diabetes"].value_counts().to_dict())

In [ ]:
#       GB raw      GB + RiskCount       GB + RiskCount + HealthyCount    GB + HealthyCount
# seed                                                                          
# 655   0.7576          0.7584                         0.7577             0.7576
# 115   0.7577          0.7580                         0.7579             0.7579
# 26    0.7578          0.7572                         0.7573             0.7574
# 760   0.7577          0.7577                         0.7574             0.7577
# 282   0.7577          0.7571                         0.7572             0.7572
# 251   0.7571          0.7568                         0.7570             0.7578
# 229   0.7573          0.7572                         0.7574             0.7573
# 143   0.7579          0.7574                         0.7572             0.7571
# 755   0.7567          0.7579                         0.7574             0.7569
# 105   0.7572          0.7580                         0.7574             0.7580
# 693   0.7575          0.7576                         0.7577             0.7576
# 759   0.7578          0.7576                         0.7577             0.7582
# 914   0.7570          0.7577                         0.7570             0.7571
# 559   0.7569          0.7579                         0.7584             0.7568
# 90    0.7569          0.7588                         0.7580             0.7577
# 605   0.7577          0.7577                         0.7579             0.7574
# 433   0.7575          0.7575                         0.7570             0.7567
# 33    0.7574          0.7577                         0.7579             0.7574
# 31    0.7564          0.7567                         0.7565             0.7566
# 96    0.7572          0.7571                         0.7576             0.7567

#                                           comparison  mean diff  wins
#                           GB + RiskCount  vs  GB raw     0.0002 12/20
#                        GB + HealthyCount  vs  GB raw     0.0000 11/20
#            GB + RiskCount + HealthyCount  vs  GB raw     0.0001 12/20
#    GB + RiskCount + HealthyCount  vs  GB + RiskCount    -0.0001 10/20
# GB + RiskCount + HealthyCount  vs  GB + HealthyCount     0.0001 11/20

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

best_gb_model = GradientBoostingClassifier(
    learning_rate=0.05,
    n_estimators=200,
    subsample=0.8,
    min_samples_leaf=50,
    max_depth=5,
    random_state=42,
)